# Sanity Check
Verify rdkit and assembly_theory toolchain works for individual molecules.

**Goal**: Test that we can compute:
1. Point group symmetry for water (should be C₂ᵥ)
2. Molecular assembly index for ethanol

**Status**: If both work, we're ready for the 100-molecule pilot.

In [ ]:
import sys
sys.path.insert(0, '/c/Users/Cicada38/Projects/math_exp')

import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

print("=" * 60)
print("SANITY CHECK: RDKit + Assembly Theory Toolchain")
print("=" * 60)
print(f"\nRDKit version: {rdkit.__version__}")

In [ ]:
# Test 1: Water point group detection
print("\n" + "=" * 60)
print("TEST 1: Water (H₂O) Point Group Detection")
print("=" * 60)

from src.compute_symmetry import compute_point_group

water_smiles = "O"
water_result = compute_point_group(water_smiles, tolerance=0.3)

print(f"\nSMILES: {water_smiles}")
print(f"Point Group: {water_result.get('point_group', 'ERROR')}")
print(f"Point Group Order: {water_result.get('order', 'ERROR')}")
print(f"Max Rotation Order: {water_result.get('max_rotation_order', 'ERROR')}")

# Water should have C2v symmetry (C2 rotation axis + 2 mirror planes)
expected_pg = "C2v"
actual_pg = water_result.get('point_group', '')
if actual_pg == expected_pg:
    print(f"✓ PASS: Water correctly identified as {expected_pg}")
else:
    print(f"⚠ CHECK: Water identified as {actual_pg} (expected {expected_pg})")
    print("  (Tolerance or algorithm may need tuning)")

In [ ]:
# Test 2: Ethanol assembly index
print("\n" + "=" * 60)
print("TEST 2: Ethanol (C₂H₆O) Molecular Assembly Index")
print("=" * 60)

from src.compute_assembly import compute_assembly_index

ethanol_smiles = "CCO"
print(f"\nSMILES: {ethanol_smiles}")

# Try API first, fallback to heuristic
ethanol_result = compute_assembly_index(ethanol_smiles, prefer="api")

print(f"Source: {ethanol_result.get('source', 'ERROR')}")
if ethanol_result.get('success'):
    ma = ethanol_result.get('assembly_index')
    print(f"Assembly Index: {ma}")
    print("✓ PASS: MA computation successful")
else:
    print(f"⚠ FALLBACK: {ethanol_result.get('error', 'Unknown error')}")
    print(f"Using heuristic instead...")
    ethanol_result = compute_assembly_index(ethanol_smiles, prefer="heuristic")
    if ethanol_result.get('success'):
        ma = ethanol_result.get('assembly_index')
        print(f"Assembly Index (heuristic): {ma}")
        print(f"Note: {ethanol_result.get('note', '')}")

In [ ]:
# Test 3: Batch computation (heuristic mode for testing)
print("\n" + "=" * 60)
print("TEST 3: Batch Computation (5 test molecules)")
print("=" * 60)

from src.compute_assembly import AssemblyIndexBatcher

test_molecules = [
    ("C", "methane"),
    ("CC", "ethane"),
    ("CCO", "ethanol"),
    ("c1ccccc1", "benzene"),
    ("C1CCCCC1", "cyclohexane")
]

batcher = AssemblyIndexBatcher(method="heuristic", delay_seconds=0.0)
smiles_list = [sm for sm, _ in test_molecules]
results = batcher.compute_batch(smiles_list)

print("\nMolecule      | SMILES       | MA    | Source")
print("-" * 60)
for (smiles, name), result in zip(test_molecules, results):
    if result.get('success'):
        ma = result.get('assembly_index')
        source = result.get('source')
        print(f"{name:13} | {smiles:12} | {ma:5} | {source}")
    else:
        print(f"{name:13} | {smiles:12} | FAIL  | {result.get('error')}")

## Summary

**If all tests pass:**
- ✓ RDKit works for symmetry detection
- ✓ MA computation works (via API or heuristic)
- → Ready for 100-molecule pilot

**If water point group is wrong:**
- Tune symmetry detection algorithm (tolerance, rotation detection)

**If ethanol MA fails:**
- Check if molecular-assembly.com API is accessible
- Heuristic fallback should still work (but is not real MA)

In [ ]:

import sys
sys.path.insert(0, '/c/Users/Cicada38/Projects/math_exp')

import rdkit
print(f"RDKit version: {rdkit.__version__}")
